In [1]:
# CELL 1: use one GPU only. Run this FIRST in a fresh session.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
n = torch.cuda.device_count()
print("GPUs visible:", n)
print("GPU name:", torch.cuda.get_device_name(0))
if n != 1:
    raise RuntimeError("Stop the session, then run this cell first again.")
print("CELL 1 DONE")


GPUs visible: 1
GPU name: Tesla T4
CELL 1 DONE


In [2]:
# CELL 2: install libraries and download SocraTeach
import sys, subprocess, os, importlib

libs = ["transformers", "peft", "trl", "datasets", "accelerate", "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + libs)

for name in libs:
    try:
        m = importlib.import_module(name)
        print("OK    ", name, getattr(m, "__version__", ""))
    except Exception as e:
        print("FAILED", name, "->", type(e).__name__, str(e)[:100])

base = "https://raw.githubusercontent.com/Ljyustc/SocraticLM/main/data/"
out = "/kaggle/working/SocraTeach_multi.json"
subprocess.run(["wget", "-q", "-O", out, base + "SocraTeach_multi.json"])
mb = os.path.getsize(out) / 1e6
print("SocraTeach_multi.json MB:", round(mb, 1))
if mb < 50:
    raise RuntimeError("Download failed. Check that Internet is On.")
print("CELL 2 DONE")


OK     transformers 5.20.2
OK     peft 0.21.2
OK     trl 1.17.1
OK     datasets 4.8.5
OK     accelerate 1.15.0
OK     bitsandbytes 0.50.2
SocraTeach_multi.json MB: 76.6
CELL 2 DONE


In [3]:
# CELL 3: build train, val and test data (equal parts, no leak)
import re, json, os, random, hashlib, collections
from datasets import load_dataset
from transformers import AutoTokenizer

SEED = 42
TRAIN_FRACTION = 0.10
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
OUT_DIR = "/kaggle/working/data"
os.makedirs(OUT_DIR, exist_ok=True)

multi = json.load(open("/kaggle/working/SocraTeach_multi.json"))
md = load_dataset("eth-nlped/mathdial")

SYSTEM = ("You are a patient Socratic math tutor. Guide the student with short questions and hints. "
          "Do not give the final answer before the student finds it. "
          "If the student makes a mistake, help them find it themselves.\n\nProblem: {q}")

def norm(t):
    return re.sub(r"\s+", " ", str(t).lower()).strip()

def qhash(q):
    return hashlib.md5(q.encode()).hexdigest()[:12]

def split_of(q):
    h = int(hashlib.md5((str(SEED) + q).encode()).hexdigest(), 16) % 100
    return "test" if h < 10 else "val" if h < 20 else "train"

def build_socrateach():
    out, skipped = [], 0
    for pid, v in multi.items():
        q = v["question"].strip()
        for did, turns in v["dialogues"].items():
            msgs, ok = [], True
            for t in turns:
                s = str(t.get("system", "")).strip()
                if not s:
                    ok = False
                    break
                msgs.append(["assistant", s, True])
                if "user" in t:
                    u = str(t["user"]).strip()
                    if not u:
                        ok = False
                        break
                    msgs.append(["user", u, False])
            if ok and len(msgs) >= 2:
                out.append({"source": "socrateach", "id": did, "question": q, "turns": msgs})
            else:
                skipped += 1
    return out, skipped

TAG = re.compile(r"\((probing|generic|telling|focus)\)\s*(.*)$", re.S)

def parse_mathdial(conv):
    turns = []
    for t in conv.split("|EOM|"):
        t = t.strip()
        if not t:
            continue
        sp, _, rest = t.partition(":")
        rest = rest.strip()
        if not rest:
            continue
        if sp.strip() == "Teacher":
            m = TAG.match(rest)
            tag, text = (m.group(1), m.group(2).strip()) if m else ("NO_TAG", rest)
            if text:
                turns.append(["assistant", text, tag != "telling"])
        else:
            turns.append(["user", rest, False])
    merged = []
    for role, text, learn in turns:
        if merged and merged[-1][0] == role:
            merged[-1][1] += "\n" + text
            merged[-1][2] = merged[-1][2] and learn
        else:
            merged.append([role, text, learn])
    return merged

def build_mathdial():
    out, skipped = [], 0
    for s in ["train", "test"]:
        for r in md[s]:
            turns = parse_mathdial(r["conversation"])
            if any(t[0] == "assistant" and t[2] for t in turns):
                out.append({"source": "mathdial", "id": "md_" + str(len(out)),
                            "question": r["question"].strip(), "turns": turns})
            else:
                skipped += 1
    return out, skipped

st_all, st_skipped = build_socrateach()
md_all, md_skipped = build_mathdial()
print("SocraTeach kept:", len(st_all), "| skipped:", st_skipped)
print("MathDial kept:", len(md_all), "| skipped:", md_skipped)

pools = {"mathdial": collections.defaultdict(list), "socrateach": collections.defaultdict(list)}
for d in md_all:
    pools["mathdial"][split_of(norm(d["question"]))].append(d)
for d in st_all:
    pools["socrateach"][split_of(norm(d["question"]))].append(d)

rng = random.Random(SEED)
chosen = {sp: [] for sp in ["train", "val", "test"]}
for sp in ["train", "val", "test"]:
    n = min(len(pools["mathdial"][sp]), len(pools["socrateach"][sp]))
    if sp == "train":
        n = max(1, int(n * TRAIN_FRACTION))
    for src in ["mathdial", "socrateach"]:
        pool = sorted(pools[src][sp], key=lambda d: d["id"])
        chosen[sp] += rng.sample(pool, n)
    print("Dialogues chosen", sp, ":", n, "from each dataset")

def make_examples(d):
    system = {"role": "system", "content": SYSTEM.format(q=d["question"])}
    exs = []
    for i, (role, text, learn) in enumerate(d["turns"]):
        if role == "assistant" and learn:
            prompt = [system] + [{"role": r, "content": t} for r, t, _ in d["turns"][:i]]
            exs.append({"source": d["source"], "dialogue_id": d["id"],
                        "qkey": qhash(norm(d["question"])),
                        "prompt": prompt,
                        "completion": [{"role": "assistant", "content": text}]})
    return exs

data = {}
for sp in ["train", "val", "test"]:
    exs = []
    for d in chosen[sp]:
        exs += make_examples(d)
    random.Random(SEED).shuffle(exs)
    data[sp] = exs
    with open(OUT_DIR + "/" + sp + ".jsonl", "w") as f:
        for e in exs:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

for sp in ["train", "val", "test"]:
    c = collections.Counter(e["source"] for e in data[sp])
    print(sp, "total:", len(data[sp]), "| mathdial:", c["mathdial"], "| socrateach:", c["socrateach"])

keys = {sp: {e["qkey"] for e in data[sp]} for sp in data}
print("Leak check:", len(keys["train"] & keys["val"]), len(keys["train"] & keys["test"]), len(keys["val"] & keys["test"]))
print("Expected: train 2388, val 3420, test 3153, leak 0 0 0")
print("CELL 3 DONE")


README.md:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/7.18M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2262 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/599 [00:00<?, ? examples/s]

SocraTeach kept: 35149 | skipped: 2
MathDial kept: 2833 | skipped: 28
Dialogues chosen train : 220 from each dataset
Dialogues chosen val : 324 from each dataset
Dialogues chosen test : 300 from each dataset
train total: 2388 | mathdial: 1223 | socrateach: 1165
val total: 3420 | mathdial: 1704 | socrateach: 1716
test total: 3153 | mathdial: 1594 | socrateach: 1559
Leak check: 0 0 0
Expected: train 2388, val 3420, test 3153, leak 0 0 0
CELL 3 DONE


In [4]:
# CELL 4: load data, model (4-bit) and LoRA settings
import json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
DATA_DIR = "/kaggle/working/data"
MAX_TOKENS = 1024

def load(sp):
    return [json.loads(l) for l in open(DATA_DIR + "/" + sp + ".jsonl")]

data = {sp: load(sp) for sp in ["train", "val", "test"]}
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def n_tokens(e):
    text = tok.apply_chat_template(e["prompt"] + e["completion"], tokenize=False)
    return len(tok(text)["input_ids"])

train_ex = [e for e in data["train"] if n_tokens(e) <= MAX_TOKENS]
val_all = [e for e in data["val"] if n_tokens(e) <= MAX_TOKENS]
val_ex = val_all[:300]
print("Train examples:", len(train_ex), "(expected 2344)")
print("Val examples used:", len(val_ex), "(expected 300)")

keep = ["prompt", "completion"]
train_ds = Dataset.from_list([{k: e[k] for k in keep} for e in train_ex])
val_ds = Dataset.from_list([{k: e[k] for k in keep} for e in val_ex])

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map={"": 0},
    dtype=torch.float16,
)
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

mods = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=mods,
)
print("CELL 4 DONE")


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

GPU memory used (GB): 2.68
CELL 4 DONE


In [5]:
# CELL 5: training settings (checks every argument name, no training here)
import dataclasses
from trl import SFTConfig

wanted = dict(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=7,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_TOKENS,
    completion_only_loss=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=42,
)

fields = {f.name for f in dataclasses.fields(SFTConfig)}
missing = [k for k in wanted if k not in fields]
print("Unsupported arguments:", missing)
if missing:
    raise RuntimeError("Stop. Send me this list.")

args = SFTConfig(**wanted)
print("Steps expected:", len(train_ds) // 16)
print("CELL 5 DONE")


Unsupported arguments: []
Steps expected: 146
CELL 5 DONE


In [7]:
import gc, time, torch
from collections import Counter
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

try:
    del trainer
except NameError:
    pass
del model
gc.collect()
torch.cuda.empty_cache()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map={"": 0},
    dtype=torch.float16,
)
print("Fresh model loaded. GPU GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

args = SFTConfig(**{**wanted, "bf16": False})
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tok,
    peft_config=lora,
)

def trainable_dtypes():
    return dict(Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))

print("Trainable weights before:", trainable_dtypes())
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)
print("Trainable weights after :", trainable_dtypes())

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
print("Minutes:", round((time.time() - t0) / 60, 1))
print("Peak GPU GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))
print("CELL 6 FIXED DONE")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Fresh model loaded. GPU GB: 5.51


Tokenizing train dataset:   0%|          | 0/2344 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2344 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2344 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2344 [00:00<?, ? examples/s]

... (3 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Trainable weights before: {'torch.bfloat16': 504}
Trainable weights after : {'torch.float32': 504}


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,1.216486,1.238808,1.235879,0.679169,279921.000000
100,1.034589,1.193618,1.166897,0.686629,562868.000000
147,1.123752,1.185076,1.182331,0.686775,813148.000000


Minutes: 41.3
Peak GPU GB: 6.95
CELL 6 FIXED DONE


In [8]:
# CELL 7: final check, save the adapter, make a zip to download
import shutil, os

ADAPTER_DIR = "/kaggle/working/lora_adapter"

metrics = trainer.evaluate()
for k, v in metrics.items():
    if isinstance(v, float):
        print("Final eval", k, "=", round(v, 4))

trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)

hist = trainer.state.log_history
print("Train loss:", [(l["step"], round(l["loss"], 3)) for l in hist if "loss" in l])
print("Eval loss:", [(l["step"], round(l["eval_loss"], 3)) for l in hist if "eval_loss" in l])

shutil.make_archive("/kaggle/working/lora_adapter", "zip", ADAPTER_DIR)
mb = os.path.getsize("/kaggle/working/lora_adapter.zip") / 1e6
print("Zip made: /kaggle/working/lora_adapter.zip", round(mb, 1), "MB")
print("IMPORTANT: download lora_adapter.zip from the Output panel before you stop the session.")
print("CELL 7 DONE")


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
1.123752,1.185076,147,1.182331,0.686775,813148.000000


Final eval eval_loss = 1.1851
Final eval eval_entropy = 1.1823
Final eval eval_mean_token_accuracy = 0.6868
Final eval eval_num_tokens = 813148.0
Train loss: [(10, 2.486), (20, 1.335), (30, 1.247), (40, 1.248), (50, 1.216), (60, 1.204), (70, 1.065), (80, 1.219), (90, 1.152), (100, 1.035), (110, 1.185), (120, 1.109), (130, 1.109), (140, 1.124)]
Eval loss: [(50, 1.239), (100, 1.194), (147, 1.185), (147, 1.185)]
Zip made: /kaggle/working/lora_adapter.zip 124.6 MB
IMPORTANT: download lora_adapter.zip from the Output panel before you stop the session.
CELL 7 DONE


In [9]:
# CELL 8: compare answers after training (same examples as before training)
m = trainer.model
m.eval()
m.config.use_cache = True

for src in ["mathdial", "socrateach"]:
    e = next(x for x in data["val"] if x["source"] == src and len(x["prompt"]) >= 4)
    text = tok.apply_chat_template(e["prompt"], tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=80, do_sample=False)
    answer = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("===== After training:", src, "=====")
    print("Last student message:", e["prompt"][-1]["content"][:250])
    print("Real teacher reply  :", e["completion"][0]["content"][:250])
    print("Model reply (after) :", answer[:400])
print("CELL 8 DONE")


===== After training: mathdial =====
Last student message: Elliott started with 10,000 steps, and he had 2,000 steps left at the end of the day. So he must have taken 8,000 steps in total. He finished half of his steps on his walks to and from school, so he took 5,000 steps during these walks. Adding the 1,0
Real teacher reply  : Good try, but remember he didn't reach 10000 steps in the day he only reach 8000. What should you be subtracting the 6000 from?
Model reply (after) : So if he had 2000 steps left at the end of the day how many steps did he take in total?
===== After training: socrateach =====
Last student message: Sure! If it took him 2.5 minutes each time, then it would take him 15 minutes to go 6 times.
Real teacher reply  : Well done! You've accurately calculated the time it would take for John to go to the bathroom 6 times based on his average time spent. So, how long would it take John to go to the bathroom 6 times based on this calculation?
Model reply (after) : Excellen